In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyod.models.knn import KNN
from pyod.models.abod import ABOD
from pyod.utils.data import generate_data
from sklearn.metrics import classification_report, confusion_matrix

# --- CONFIGURATION ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# --- 1. DATA GENERATION (BENCHMARK ENVIRONMENT) ---
print("Generating controlled synthetic benchmark data...")
# Simulating a dataset with 10% anomalies (e.g., fraudulent transactions or network intrusions)
X_train, X_test, y_train, y_test = generate_data(
    n_train=1000,
    n_test=200,
    n_features=2,
    contamination=0.1,
    random_state=42
)

print(f"Training data shape: {X_train.shape} | Anomalies: {int(0.1 * 1000)}")
print(f"Testing data shape:  {X_test.shape}  | Anomalies: {int(0.1 * 200)}")

# --- 2. MODEL INITIALIZATION & TRAINING ---
print("\nInitializing models for comparative analysis...")

# Model A: K-Nearest Neighbors (Distance-based Outlier Detection)
print("Training KNN Model...")
knn_model = KNN(n_neighbors=5, method='largest')
knn_model.fit(X_train)

# Model B: Angle-Based Outlier Detection (Variance-based)
print("Training ABOD Model...")
abod_model = ABOD(contamination=0.1)
abod_model.fit(X_train)

# --- 3. PREDICTIONS ---
# 0 = Normal (Inlier), 1 = Anomaly (Outlier)
y_test_pred_knn = knn_model.predict(X_test)
y_test_pred_abod = abod_model.predict(X_test)

# --- 4. EVALUATION METRICS ---
print("\n" + "="*50)
print("PERFORMANCE REPORT: K-Nearest Neighbors (KNN)")
print("="*50)
print(classification_report(y_test, y_test_pred_knn, target_names=['Normal', 'Anomaly']))

print("\n" + "="*50)
print("PERFORMANCE REPORT: Angle-Based Outlier Detection (ABOD)")
print("="*50)
print(classification_report(y_test, y_test_pred_abod, target_names=['Normal', 'Anomaly']))

# --- 5. VISUALIZATION: INLIERS VS OUTLIERS ---
def plot_anomalies(X, y_pred, title, ax):
    """Plots the decision boundary results for anomaly detection."""
    # Plot normal data points
    ax.scatter(X[y_pred == 0, 0], X[y_pred == 0, 1],
               c='steelblue', label='Predicted Normal', alpha=0.7, edgecolors='w', s=60)
    # Plot anomalous data points
    ax.scatter(X[y_pred == 1, 0], X[y_pred == 1, 1],
               c='crimson', label='Predicted Anomaly', alpha=0.9, edgecolors='k', marker='X', s=80)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc='upper left')

print("\nGenerating Decision Boundary Plots...")
fig, axes = plt.subplots(1, 2)

plot_anomalies(X_test, y_test_pred_knn, "Distance-Based Anomaly Detection (KNN)", axes[0])
plot_anomalies(X_test, y_test_pred_abod, "Angle-Based Anomaly Detection (ABOD)", axes[1])

plt.tight_layout()
plt.show()